# Advanced debugging view for the RAG pipeline

This notebook is for deeper inspection of the graph and retrieval behavior.
It shows:
- the ingest graph structure
- the ingestion report
- matched chunk metadata
- retrieval scores and content previews
- a few query variations to compare results

This uses the offline `hash` embedding mode so it runs without any live API key.

In [1]:
from pathlib import Path
import sys

root = Path.cwd()
print(f"Working directory: {root}")
assert (root / 'veriti_rag').is_dir(), 'Open this notebook from the project root directory.'
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
root

Working directory: c:\Projects\New folder\veriti_rag_app\veriti_rag_app


WindowsPath('c:/Projects/New folder/veriti_rag_app/veriti_rag_app')

In [2]:
from veriti_rag.config import Settings
from veriti_rag.models import HashEmbeddings
from veriti_rag.store import get_vectorstore
from veriti_rag.ingest import build_ingest_graph

s = Settings(
    kb_dir=str(root / 'knowledge_base'),
    index_dir=str(root / 'vector_index'),
    embeddings='hash',
    client_id='maven_demo',
)
vs = get_vectorstore(s, HashEmbeddings())
graph = build_ingest_graph(s, vs)

g = graph.get_graph()
print('Graph object:', graph)
print('Node names:', list(g.nodes))
print('Edges:', [(e.source, e.target) for e in g.edges])

Graph object: <langgraph.graph.state.CompiledStateGraph object at 0x000001F32437C980>
Node names: ['__start__', 'load', 'split', 'index', '__end__']
Edges: [('__start__', 'load'), ('load', 'split'), ('split', 'index'), ('index', '__end__')]


## Ingest and inspect the report

This is the same ingestion pipeline used by the app, but we print the result object in a way that is easier to inspect during debugging.

In [3]:
from veriti_rag.ingest import ingest_all

reports = ingest_all(s, vs)
for report in reports:
    print(report)
    print('-' * 80)

{'client': 'maven_demo', 'chunks': 33, 'added': 0, 'deleted': 0, 'unchanged': 33}
--------------------------------------------------------------------------------


In [4]:
def debug_search(query: str, client_id: str = 'maven_demo', k: int = 5):
    from veriti_rag.store import search
    rows = []
    for doc, score in search(vs, query, client_id, k=k):
        rows.append({
            'query': query,
            'score': round(float(score), 4),
            'source': doc.metadata.get('source'),
            'doc_type': doc.metadata.get('doc_type'),
            'rule_id': doc.metadata.get('rule_id'),
            'chunk_id': doc.metadata.get('chunk_id'),
            'client_id': doc.metadata.get('client_id'),
            'preview': doc.page_content[:300].replace('\n', ' '),
        })
    return rows

print('debug_search ready')

debug_search ready


## Retrieval debugging

Run a query and inspect each matched chunk. The output includes score, metadata, and a short text preview.

In [5]:
query = 'won deals missing close value'
results = debug_search(query, k=3)
print(f"Query: {query}\n")
for row in results:
    print('---')
    for key, value in row.items():
        print(f'{key}: {value}')

Query: won deals missing close value

---
query: won deals missing close value
score: 0.4794
source: drc_rules.yml
doc_type: drc_rule
rule_id: DRC-COMP-002
chunk_id: fd98bee2eb517016
client_id: maven_demo
preview: id: DRC-COMP-002 description: Won deals must have a close value table: sales_pipeline column: close_value dimension: completeness severity: critical bsd: revenue_reporting fail_condition: deal_stage = 'Won' AND close_value IS NULL
---
query: won deals missing close value
score: 0.3789
source: remediation_guide.md
doc_type: remediation
rule_id: DRC-COMP-002
chunk_id: be7f445781873dff
client_id: maven_demo
preview: # Remediation guide   ## DRC-COMP-002 Missing close value on won deals Back-fill from the billing or contract system. If unavailable, flag the deal for the sales owner to confirm and exclude it from revenue reports until resolved. Add a required-field rule at data entry.
---
query: won deals missing close value
score: 0.2992
source: business_rules.md
doc_type: busine

In [6]:
queries = [
    'won deals missing close value',
    'close value',
    'sales pipeline',
    'DRC-COMP-002',
    'won deals',
]

for q in queries:
    print('\n' + '=' * 100)
    print(f'QUERY: {q}')
    matches = debug_search(q, k=3)
    if not matches:
        print('No matches found')
        continue
    for row in matches:
        print(f"score={row['score']} | source={row['source']} | rule_id={row['rule_id']} | doc_type={row['doc_type']}")
        print(row['preview'])
        print('-' * 40)


QUERY: won deals missing close value
score=0.4794 | source=drc_rules.yml | rule_id=DRC-COMP-002 | doc_type=drc_rule
id: DRC-COMP-002 description: Won deals must have a close value table: sales_pipeline column: close_value dimension: completeness severity: critical bsd: revenue_reporting fail_condition: deal_stage = 'Won' AND close_value IS NULL
----------------------------------------
score=0.3789 | source=remediation_guide.md | rule_id=DRC-COMP-002 | doc_type=remediation
# Remediation guide   ## DRC-COMP-002 Missing close value on won deals Back-fill from the billing or contract system. If unavailable, flag the deal for the sales owner to confirm and exclude it from revenue reports until resolved. Add a required-field rule at data entry.
----------------------------------------
score=0.2992 | source=business_rules.md | rule_id=None | doc_type=business_rule
# Metric definitions   ## Win rate Win rate = Won deals divided by closed deals (Won plus Lost). Open deals (Prospecting, Engagin

## Quick comparison view

This last cell lets you test more queries and compare which documents are retrieved for each one.

In [7]:
custom_queries = [
    'approved sales reps',
    'risk rule',
    'contract value',
    'data quality rule',
]

for q in custom_queries:
    print(f'\n=== QUERY: {q} ===')
    rows = debug_search(q, k=2)
    print('match_count=', len(rows))
    for row in rows:
        print(row['score'], row['source'], row['rule_id'], row['doc_type'])


=== QUERY: approved sales reps ===
match_count= 2
0.1747 data_dictionary.csv None data_dictionary
0.1656 schema.yml None schema

=== QUERY: risk rule ===
match_count= 2
0.0902 bsd_definitions.md None bsd_definition
0.0716 bsd_definitions.md None bsd_definition

=== QUERY: contract value ===
match_count= 2
0.139 data_dictionary.csv None data_dictionary
0.0902 business_rules.md None business_rule

=== QUERY: data quality rule ===
match_count= 2
0.1297 remediation_guide.md DRC-COMP-002 remediation
0.0563 data_dictionary.csv None data_dictionary
